In [75]:
from datasets import load_dataset
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [27]:
data = load_dataset("fancyzhx/ag_news")

In [28]:
# load_dataset returns a datasetDict which containes multiple splits such as: train and test, and it does not have
# .to_pandas() methods. So, you have to convert each split alone. 
print(data)
train_dataset = data["train"]
test_dataset = data["test"]
print(train_dataset[0])
print(test_dataset[0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
{'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.", 'label': 2}


### Data Preprocessing 

In [49]:
# Convert the train split to pandas
train_df = data['train'].to_pandas()
test_df = data['test'].to_pandas()

In [84]:
print(train_df.shape)
print("===========")
print(test_df.shape)

(120000, 2)
(7600, 2)


In [50]:
print(train_df.head())
print(test_df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2
                                                text  label
0  Fears for T N pension after talks Unions repre...      2
1  The Race is On: Second Private Team Sets Launc...      3
2  Ky. Company Wins Grant to Study Peptides (AP) ...      3
3  Prediction Unit Helps Forecast Wildfires (AP) ...      3
4  Calif. Aims to Limit Farm-Related Smog (AP) AP...      3


In [78]:
train_df.isnull().sum()

text     0
label    0
dtype: int64

In [70]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    120000 non-null  str  
 1   label   120000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 28.9 MB


In [ ]:
# Training 
X_train = np.array(train_df.iloc[:, 0]) # Text
print(X_train[0])
y_train = np.array(train_df.iloc[:, 1]) # Label 
print(y_train[0])

# Testing 
X_test = np.array(test_df.iloc[:, 0])
print(X_test[0])
y_test = np.array(test_df.iloc[:, 1])
print(y_test[0])

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
2


In [ ]:
# The data is already splitted into training and testing.
# But we need to split the testing dataset into validation data and testing data.

# Split the test data into testing and validation 
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5)

### Tokenize the text

- PyTorch can not convert raw text into tensors, it should be converted into nubmers first, then tokenized.

### Neural Network 

In [73]:
class AGNewsDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long).to(device) # torch.long because its text not numbers
        self.y = torch.tensor(y, dtype=torch.long).to(device)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [ ]:
training_data = AGNewsDataset(X_train, y_train)
validation_data = AGNewsDataset(X_val, y_val)
testing_data = AGNewsDataset(X_test, y_test)

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.